
# 02 — ICD-Based Low Back Pain Phenotyping

**Project:** Predicting Opioid Dispensing in Low Back Pain Emergency Visits  
**Phase:** 02 — ICD Clinical Phenotyping  
**Dataset:** MIMIC-IV-ED v2.2  
**Unit of Analysis:** ED encounter (`stay_id`)  
**Status:** Operational phenotype definitions under final review

## Executive Summary

This notebook documents the development and quality assessment of two ICD-based low back pain (LBP) phenotype definitions for a retrospective emergency department machine learning study.

Using a previously extracted MIMIC-IV-ED diagnosis inventory, literature-informed code classification, and a versioned phenotype codebook, the analysis evaluates a five-code strict LBP definition (C1) and a twelve-code expanded definition (C2).

The primary output is a reproducible ICD codebook for encounter-level cohort construction. The analysis focuses on code selection, evidence traceability, and operational consistency rather than independent clinical validation.

**Status:** Operational phenotype codebook v1.0 finalized; independent clinical validation not performed.

## Table of Contents

1. [Study Context & Phenotype Scope](#1-study-context--phenotype-scope)
2. [ICD Inventory & Candidate Identification](#2-icd-inventory--candidate-identification)
3. [Evidence Review & Rule-Based Clinical Classification](#3-evidence-review--rule-based-clinical-classification)
4. [Codebook Construction & Quality Assurance](#4-codebook-construction--quality-assurance)
5. [Final Phenotype Definitions & Phase 4 Handoff](#5-final-phenotype-definitions--phase-4-handoff)


In [1]:

from pathlib import Path

import numpy as np
import pandas as pd

# Project paths
PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RESULTS_DIR = PROJECT_ROOT / "results" / "02_icd_phenotyping"
CODEBOOK_DIR = PROJECT_ROOT / "codebooks"

# Input files
INVENTORY_PATH = RESULTS_DIR / "icd_inventory_raw.csv"


# Display configuration
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)


print("P2 Global Setup verified.")


P2 Global Setup verified.



## 1. Study Context & Phenotype Scope

### 1.1 Research Context

Low back pain (LBP) is a common clinical presentation that can reflect different underlying conditions. Its identification in administrative healthcare data is challenging because ICD diagnosis codes may represent overlapping symptoms, anatomical regions, and clinical presentations (Hartvigsen et al., 2018).

This project uses ICD codes as operational indicators of LBP-related ED encounters rather than definitive evidence of clinically confirmed nonspecific low back pain.

The central methodological concern is that different code-selection rules may identify different study populations, potentially influencing observed opioid dispensing prevalence and downstream predictive model performance.

### 1.2 Phenotype Definitions

Two prespecified ICD-based phenotypes are used:

| Phenotype | Definition | Research Role |
|---|---|---|
| **C1 — Strict LBP** | Five codes representing explicit low back pain, including LBP with sciatica | Primary cohort |
| **C2 — Expanded LBP** | Twelve codes including C1, additional sciatica-only codes, and broader back pain symptom codes | Sensitivity cohort |

C1 is nested within C2. The additional encounters identified by C2 will be examined separately during cohort construction and sensitivity analysis.

Neither definition establishes clinically confirmed nonspecific LBP.

### 1.3 Objectives and Scope

This notebook aims to:

1. Document the previously extracted ICD diagnosis inventory.
2. Review the literature-informed selection of twelve candidate phenotype codes.
3. Validate codebook structure, evidence traceability, and C1/C2 consistency.
4. Specify reproducible ICD matching rules for encounter-level cohort construction.

The primary unit of downstream analysis is the ED encounter (`stay_id`). Multiple diagnosis records for the same encounter will be aggregated to encounter-level phenotype indicators in P4.

This notebook does not perform clinical chart validation, opioid outcome construction, or predictive modeling.

### 1.4 Literature Context

The literature review focuses on the challenges of defining LBP populations using clinical and administrative data.

- Hartvigsen et al. (2018) provides clinical context for the heterogeneity of LBP.
- *Identifying neck and back pain in administrative data: defining the right cohort* (2012) motivates careful cohort selection.
- *Sensitivity and specificity of algorithms for the identification of nonspecific low back pain in medico-administrative databases* (2023) highlights the importance of evaluating administrative case-identification definitions.

These sources motivate the phenotype design but do not independently validate the specific C1/C2 code lists used in this project.

### 1.5 Expected Outputs

The outputs are:

- A reviewed, versioned ICD phenotype codebook.
- Validated C1 and C2 code membership rules.
- A documented operational definition for encounter-level matching.
- Explicit limitations and integration requirements for P4.

No patient-level MIMIC data are intended for publication in the repository.

### References

1. Hartvigsen, J., et al. (2018). *What low back pain is and why we need to pay attention.* The Lancet. https://pubmed.ncbi.nlm.nih.gov/29573870/
2. *Identifying neck and back pain in administrative data: defining the right cohort.* (2012). https://pubmed.ncbi.nlm.nih.gov/22127268/
3. *Sensitivity and specificity of algorithms for the identification of nonspecific low back pain in medico-administrative databases.* (2023). https://pubmed.ncbi.nlm.nih.gov/36728479/



## 2. ICD Inventory & Candidate Identification

### 2.1 ICD Diagnosis Inventory

**Objective:** Verify the source ICD diagnosis inventory used for rule-based candidate identification.

**Method:** A previously executed BigQuery query extracts distinct `(icd_version, icd_code)` combinations from MIMIC-IV-ED `diagnosis`, together with recorded descriptions and frequency summaries.

**SQL source:** `sql/02_icd_phenotyping/01_extract_icd_inventory.sql`

**Input:** `results/02_icd_phenotyping/icd_inventory_raw.csv`

The inventory includes:

- `diagnosis_count`: Number of diagnosis records.
- `visit_count`: Number of distinct ED encounters associated with a code.
- `patient_count`: Number of distinct patients associated with a code.

ICD codes are preserved as strings, and ICD version is treated as part of the unique identifier.

The inventory supports candidate identification but does not itself determine clinical eligibility. Encounter-level cohort counts will be computed separately in P4.


In [2]:

# Load raw ICD diagnosis inventory
inventory = pd.read_csv(
    INVENTORY_PATH,
    dtype={
        "icd_version": "string",
        "icd_code": "string"
    }
)

# Validate source identifiers
required_columns = {
    "icd_version", "icd_code", "ed_description",
    "diagnosis_count", "visit_count", "patient_count"
}

assert required_columns.issubset(inventory.columns)
assert inventory["icd_version"].isin(["9", "10"]).all()
assert inventory[["icd_version", "icd_code"]].notna().all().all()
assert not inventory.duplicated(
    ["icd_version", "icd_code"]
).any()

# Summarize available codes by ICD version
inventory_summary = (
    inventory.groupby("icd_version", as_index=False)
    .agg(
        unique_codes=("icd_code", "size"),
        diagnosis_records=("diagnosis_count", "sum")
    )
)

display(inventory_summary)

print(f"Total code/version pairs: {len(inventory):,}")
print(f"Total diagnosis records: {inventory['diagnosis_count'].sum():,}")


,icd_version,unique_codes,diagnosis_records
0,10,8526,456035
1,9,4684,443015


Total code/version pairs: 13,210
Total diagnosis records: 899,050



#### Interpretation

The diagnosis inventory contains 13,210 unique ICD code/version combinations representing 899,050 diagnosis records.

The inventory provides a reproducible starting point for identifying codes potentially relevant to low back pain. However, administrative diagnosis descriptions alone are insufficient to determine whether an encounter represents clinically confirmed nonspecific LBP.

Per-code encounter counts cannot be summed to obtain unique cohort sizes because an ED encounter may contain multiple diagnosis codes.



### 2.2 Clinical Concept & Candidate Screening Rules

**Objective:** Define transparent screening criteria for identifying ICD codes potentially relevant to the study's low back pain phenotypes.

**Clinical concept framework**

The study considers three symptom-based concepts:

1. **Explicit low back pain:** Codes describing low back pain or lumbago, including lumbago with sciatica.
2. **Sciatica:** Codes describing sciatica without explicitly identifying low back pain.
3. **Broader back pain:** Codes describing backache or dorsalgia without specifying the lumbar region.

The first concept is intended for the primary C1 phenotype. The expanded C2 phenotype may additionally include the second and third concepts.

Codes describing a specific underlying pathology, injury, or unrelated anatomical condition are not automatically eligible merely because their descriptions contain relevant anatomical terms.

**Candidate screening approach**

A case-insensitive keyword search is applied to the diagnosis inventory using:

- `low back`
- `lumbago`
- `sciatica`
- `backache`
- `dorsalgia`

These terms are screening rules, not final inclusion rules.

The search is intended to identify a manageable set of symptom-related candidates for targeted manual review. It is not an exhaustive search of every possible diagnosis associated with LBP, and relevant codes not containing these terms may be missed.

Final eligibility and cohort assignments will be determined through documented clinical concept review and ICD terminology verification in Section 3.



### 2.3 Candidate Identification

**Objective:** Apply the prespecified keyword screening rules to the source ICD inventory and generate a candidate list for clinical concept review.

**Method:** Search the recorded ICD descriptions for the five specified symptom-related terms using case-insensitive regular-expression matching.

All matching codes are retained at this stage, including potentially ineligible codes. No clinical inclusion or exclusion decisions are made by the keyword search.

The resulting candidate list will serve as the input for the evidence review and rule-based classification in Section 3.


In [3]:

# Prespecified symptom-related screening terms
SCREENING_TERMS = [
    "low back",
    "lumbago",
    "sciatica",
    "backache",
    "dorsalgia"
]

screening_pattern = "|".join(SCREENING_TERMS)

# Identify potentially relevant codes
candidates = inventory.loc[
    inventory["ed_description"].str.contains(
        screening_pattern,
        case=False,
        regex=True,
        na=False
    )
].copy()

candidates = candidates.sort_values(
    ["icd_version", "icd_code"]
).reset_index(drop=True)

# Basic candidate integrity checks
assert not candidates.duplicated(
    ["icd_version", "icd_code"]
).any()

print(f"Candidate codes identified: {len(candidates)}")

display(
    candidates[
        [
            "icd_version",
            "icd_code",
            "ed_description"
        ]
    ]
)


Candidate codes identified: 15


,icd_version,icd_code,ed_description
0,10,M5430,"Sciatica, unspecified side"
1,10,M5431,"Sciatica, right side"
2,10,M5432,"Sciatica, left side"
3,10,M5440,"Lumbago with sciatica, unspecified side"
4,10,M5441,"Lumbago with sciatica, right side"
5,10,M5442,"Lumbago with sciatica, left side"
6,10,M545,Low back pain
7,10,M5489,Other dorsalgia
8,10,M549,"Dorsalgia, unspecified"
9,10,S31010A,"Lac w/o fb of low back and pelv w/o penet retroperiton, init"



#### Interpretation

The keyword screening identified 15 ICD code/version combinations from the MIMIC-IV-ED diagnosis inventory.

The candidate set includes codes explicitly describing low back pain, sciatica, and broader back pain symptoms, as well as injury codes containing the phrase "low back."

This demonstrates an important limitation of description-based screening: lexical relevance does not necessarily imply eligibility for a symptom-defined clinical phenotype.

The candidate list will undergo a targeted, rule-based review in Section 3 to determine which codes meet the operational definitions of C1 and C2.

No code is assigned final cohort membership at this screening stage.



## 3. Evidence Review & Rule-Based Clinical Classification

### 3.1 Clinical Concepts, Evidence & Historical Coding

**Objective:** Establish literature-informed, rule-based criteria for classifying candidate ICD codes into strict and expanded low back pain (LBP) phenotypes.

**Evidence Framework**

The classification is informed by three complementary sources:

1. **Clinical concepts:** Explicit LBP, LBP with sciatica, sciatica alone, and broader back pain presentations.
2. **ICD terminology:** ICD-9-CM and ICD-10-CM code descriptions and their historical meanings.
3. **Published research:** Studies examining the challenges of identifying back pain populations using administrative diagnosis data.

Hartvigsen et al. (2018) provides clinical context for the heterogeneity of LBP. Administrative cohort studies (2012; 2023) motivate careful definition of code-based study populations.

These publications support the methodological approach but do not independently validate the specific code assignments used here.

**Operational Classification Rules**

| Category | Inclusion / Exclusion Rule | Assignment |
|---|---|---|
| CORE_LBP | Explicit low back pain or lumbago | C1 + C2 |
| LBP_WITH_SCIATICA | Explicit lumbago with sciatica | C1 + C2 |
| SCIATICA_ONLY | Sciatica without explicit LBP | C2 only |
| EXPANDED_BACK_PAIN | Backache or dorsalgia without explicit lumbar localization | C2 only |
| EXCLUDED_INJURY | Specific traumatic injury rather than a back pain symptom code | Excluded |

C1 is defined as a narrower symptom-based phenotype. C2 includes C1 and additional symptom codes to evaluate sensitivity to broader case definitions.

**Historical Coding Considerations**

MIMIC-IV-ED v2.2 covers ED encounters from 2011–2019, spanning the U.S. transition from ICD-9-CM to ICD-10-CM on October 1, 2015.

Codes are therefore identified by the composite key (`icd_version`, `icd_code`). Historical terminology is used when interpreting the recorded diagnoses.

ICD-10-CM undergoes periodic updates. For example, `M54.5` represented low back pain during the MIMIC-IV-ED study period but was replaced by more specific codes effective October 1, 2021, after the dataset's collection period.

The present study does not reconstruct every annual ICD-10-CM release or automatically add codes introduced after the observation period.

**Decision Principles**

- Keyword matching identifies candidates but does not determine inclusion.
- Final classification is based on the documented meaning of each code and the study's clinical concept rules.
- Codes describing specific injuries are not included solely because they mention the low back.
- ICD-defined phenotypes are operational research definitions, not independently validated clinical diagnoses.



### 3.2 Rule-Based Code Classification

**Objective:** Assign the 15 candidate ICD codes to predefined clinical categories and determine their eligibility for the C1 and C2 phenotypes.

**Method:** Each candidate code is reviewed against its recorded ICD description and the clinical concept rules established in Section 3.1.

Classification decisions are explicitly documented in a code-level mapping rather than automatically inferred from keywords.

The classification distinguishes:

- Explicit LBP and lumbago with sciatica, included in both C1 and C2.
- Sciatica-only and broader back pain symptoms, included in C2 only.
- Injury-specific codes, excluded from both phenotypes.

Python implements these researcher-defined decisions and checks that every screened candidate receives exactly one classification.

The resulting assignments represent a targeted operational review, not independent clinical adjudication.


In [4]:

# Explicit code-level classification decisions
# ICD version and code are treated as a composite identifier.

CODE_DECISIONS = {
    # Explicit low back pain
    ("9", "7242"): "CORE_LBP",
    ("10", "M545"): "CORE_LBP",

    # Lumbago with sciatica
    ("10", "M5440"): "LBP_WITH_SCIATICA",
    ("10", "M5441"): "LBP_WITH_SCIATICA",
    ("10", "M5442"): "LBP_WITH_SCIATICA",

    # Sciatica without explicit LBP
    ("9", "7243"): "SCIATICA_ONLY",
    ("10", "M5430"): "SCIATICA_ONLY",
    ("10", "M5431"): "SCIATICA_ONLY",
    ("10", "M5432"): "SCIATICA_ONLY",

    # Broader back pain symptoms
    ("9", "7245"): "EXPANDED_BACK_PAIN",
    ("10", "M5489"): "EXPANDED_BACK_PAIN",
    ("10", "M549"): "EXPANDED_BACK_PAIN",

    # Specific traumatic injuries
    ("10", "S31010A"): "EXCLUDED_INJURY",
    ("10", "S31040A"): "EXCLUDED_INJURY",
    ("10", "S31050A"): "EXCLUDED_INJURY",
}

CATEGORY_RATIONALE = {
    "CORE_LBP": "Explicit low back pain or lumbago.",
    "LBP_WITH_SCIATICA": "Explicit lumbago with sciatica.",
    "SCIATICA_ONLY": (
        "Sciatica without explicit low back pain; "
        "expanded phenotype only."
    ),
    "EXPANDED_BACK_PAIN": (
        "Back pain without explicit lumbar localization; "
        "expanded phenotype only."
    ),
    "EXCLUDED_INJURY": (
        "Specific traumatic injury rather than a "
        "back pain symptom code."
    ),
}

# Construct the researcher-defined decision table
decision_df = pd.DataFrame([
    {
        "icd_version": version,
        "icd_code": code,
        "clinical_category": category,
        "clinical_rationale": CATEGORY_RATIONALE[category],
    }
    for (version, code), category in CODE_DECISIONS.items()
])

# Ensure decisions correspond exactly to screened candidates
candidate_keys = set(zip(
    candidates["icd_version"],
    candidates["icd_code"]
))
decision_keys = set(CODE_DECISIONS)

assert candidate_keys == decision_keys, (
    f"Unreviewed candidates: {candidate_keys - decision_keys}; "
    f"Unmatched decisions: {decision_keys - candidate_keys}"
)

# Attach clinical decisions to the source inventory
reviewed_codes = candidates.merge(
    decision_df,
    on=["icd_version", "icd_code"],
    how="left",
    validate="one_to_one"
)

# Assign cohort membership using category-level rules
C1_CATEGORIES = {"CORE_LBP", "LBP_WITH_SCIATICA"}
C2_CATEGORIES = C1_CATEGORIES | {
    "SCIATICA_ONLY",
    "EXPANDED_BACK_PAIN"
}

reviewed_codes["c1_include"] = (
    reviewed_codes["clinical_category"].isin(C1_CATEGORIES)
)

reviewed_codes["c2_include"] = (
    reviewed_codes["clinical_category"].isin(C2_CATEGORIES)
)

# Summarize classification decisions
classification_summary = (
    reviewed_codes.groupby("clinical_category", as_index=False)
    .agg(
        candidate_codes=("icd_code", "size"),
        c1_codes=("c1_include", "sum"),
        c2_codes=("c2_include", "sum")
    )
)

display(classification_summary)

display(
    reviewed_codes[
        [
            "icd_version",
            "icd_code",
            "ed_description",
            "clinical_category",
            "clinical_rationale",
            "c1_include",
            "c2_include"
        ]
    ].sort_values(["icd_version", "icd_code"])
)


,clinical_category,candidate_codes,c1_codes,c2_codes
0,CORE_LBP,2,2,2
1,EXCLUDED_INJURY,3,0,0
2,EXPANDED_BACK_PAIN,3,0,3
3,LBP_WITH_SCIATICA,3,3,3
4,SCIATICA_ONLY,4,0,4


,icd_version,icd_code,ed_description,clinical_category,clinical_rationale,c1_include,c2_include
0,10,M5430,"Sciatica, unspecified side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True
1,10,M5431,"Sciatica, right side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True
2,10,M5432,"Sciatica, left side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True
3,10,M5440,"Lumbago with sciatica, unspecified side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True
4,10,M5441,"Lumbago with sciatica, right side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True
5,10,M5442,"Lumbago with sciatica, left side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True
6,10,M545,Low back pain,CORE_LBP,Explicit low back pain or lumbago.,True,True
7,10,M5489,Other dorsalgia,EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True
8,10,M549,"Dorsalgia, unspecified",EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True
9,10,S31010A,"Lac w/o fb of low back and pelv w/o penet retroperiton, init",EXCLUDED_INJURY,Specific traumatic injury rather than a back pain symptom code.,False,False



### 3.3 Classification Results, Evidence & Limitations

**Classification Results**

The targeted review classified 15 candidate ICD code/version combinations into five operational categories.

Twelve codes were selected for the expanded LBP phenotype (C2), including five codes representing explicit low back pain or lumbago with sciatica (C1). The additional seven C2 codes represent sciatica-only or broader back pain presentations.

Three injury-specific codes were excluded because they describe traumatic injuries rather than back pain symptom diagnoses.

**Evidence Interpretation**

The classification uses recorded ICD terminology and prespecified clinical concept rules to document code-level decisions.

Published administrative phenotyping studies support the importance of transparent and carefully defined case-identification algorithms. However, these studies do not independently validate the specific C1 and C2 code lists.

The resulting definitions should therefore be described as *literature-informed, researcher-defined operational phenotypes* rather than clinically validated diagnostic algorithms.

**Limitations**

1. The keyword-based candidate search is targeted rather than exhaustive.
2. Recorded ICD descriptions may not capture all clinical nuances or historical coding conventions.
3. The study does not reconstruct all annual ICD-10-CM revisions.
4. Included encounters may contain additional diagnoses indicating underlying pathology.
5. No independent chart review or clinical reference standard is available to estimate phenotype sensitivity or specificity.

**Implications for Codebook Construction**

The reviewed classification table provides the input for Section 4, where code uniqueness, version matching, cohort membership, and completeness will be validated before exporting the final codebook.

Cohort sizes will be calculated in P4 using unique ED encounters (`stay_id`), rather than by summing per-code diagnosis frequencies.

### References

1. Hartvigsen, J., et al. (2018). *What low back pain is and why we need to pay attention.* The Lancet. https://pubmed.ncbi.nlm.nih.gov/29573870/
2. *Identifying neck and back pain in administrative data: defining the right cohort.* (2012). https://pubmed.ncbi.nlm.nih.gov/22127268/
3. *Sensitivity and specificity of algorithms for the identification of nonspecific low back pain in medico-administrative databases.* (2023). https://pubmed.ncbi.nlm.nih.gov/36728479/
4. Centers for Disease Control and Prevention. *ICD-10-CM Official Coding Guidelines and Code Files.* https://www.cdc.gov/nchs/icd/icd-10-cm/



## 4. Codebook Construction & Quality Assurance

### 4.1 Codebook Construction

**Objective:** Construct a reproducible ICD phenotype codebook from the researcher-defined classification decisions established in Section 3.

**Method:** Select symptom-related codes from the reviewed candidate set, retain their ICD identifiers and descriptions, and assign C1/C2 cohort membership based on the documented clinical concept rules.

The codebook includes:

- ICD version and code.
- Recorded ICD description.
- Clinical classification category.
- Inclusion rationale.
- C1 Strict LBP membership.
- C2 Expanded LBP membership.
- Codebook version and verification status.

Injury-specific codes are excluded from the final phenotype codebook but remain documented in the Section 3 classification audit.

The codebook represents operational research definitions rather than clinically validated diagnoses.


In [5]:

# Construct the operational phenotype codebook
codebook_v1 = (
    reviewed_codes.loc[
        reviewed_codes["c2_include"],
        [
            "icd_version",
            "icd_code",
            "ed_description",
            "clinical_category",
            "clinical_rationale",
            "c1_include",
            "c2_include"
        ]
    ]
    .copy()
    .sort_values(["icd_version", "icd_code"])
    .reset_index(drop=True)
)

# Add versioning and review metadata
codebook_v1["codebook_version"] = "v1.0-draft"
codebook_v1["evidence_basis"] = (
    "Historical ICD terminology and prespecified "
    "clinical concept rules"
)
codebook_v1["source_verification_status"] = "pending_review"

display(codebook_v1)


,icd_version,icd_code,ed_description,clinical_category,clinical_rationale,c1_include,c2_include,codebook_version,evidence_basis,source_verification_status
0,10,M5430,"Sciatica, unspecified side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
1,10,M5431,"Sciatica, right side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
2,10,M5432,"Sciatica, left side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
3,10,M5440,"Lumbago with sciatica, unspecified side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
4,10,M5441,"Lumbago with sciatica, right side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
5,10,M5442,"Lumbago with sciatica, left side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
6,10,M545,Low back pain,CORE_LBP,Explicit low back pain or lumbago.,True,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
7,10,M5489,Other dorsalgia,EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
8,10,M549,"Dorsalgia, unspecified",EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review
9,9,7242,LUMBAGO,CORE_LBP,Explicit low back pain or lumbago.,True,True,v1.0-draft,Historical ICD terminology and prespecified clinical concept rules,pending_review



### 4.2 Codebook Quality Assurance

**Objective:** Verify that the generated phenotype codebook is structurally consistent and implements the intended C1/C2 definitions.

**Validation checks:**

1. All required fields are present and non-null.
2. Each (`icd_version`, `icd_code`) combination is unique.
3. ICD versions use the expected identifiers (`9` and `10`).
4. All included codes are present in the original MIMIC-IV-ED diagnosis inventory.
5. C1 contains five codes, C2 contains twelve codes, and C1 is nested within C2.
6. No injury-specific codes are included.

These checks establish technical and operational consistency. They do not independently verify historical ICD terminology or clinical phenotype accuracy.


In [6]:

# 1. Required fields and identifiers
required_fields = [
    "icd_version",
    "icd_code",
    "ed_description",
    "clinical_category",
    "clinical_rationale",
    "c1_include",
    "c2_include",
    "codebook_version",
    "evidence_basis",
    "source_verification_status"
]

assert codebook_v1[required_fields].notna().all().all()
assert not codebook_v1.duplicated(
    ["icd_version", "icd_code"]
).any()

# 2. ICD version validity
assert codebook_v1["icd_version"].isin(["9", "10"]).all()

# 3. Verify matching to source inventory
source_check = codebook_v1.merge(
    inventory[["icd_version", "icd_code"]],
    on=["icd_version", "icd_code"],
    how="left",
    indicator=True,
    validate="one_to_one"
)

assert source_check["_merge"].eq("both").all()

# 4. Cohort structure and nesting
assert len(codebook_v1) == 12
assert codebook_v1["c1_include"].sum() == 5
assert codebook_v1["c2_include"].sum() == 12

assert (
    ~codebook_v1["c1_include"]
    | codebook_v1["c2_include"]
).all()

# 5. Check classification rules
assert codebook_v1.loc[
    codebook_v1["c1_include"], "clinical_category"
].isin(C1_CATEGORIES).all()

assert codebook_v1["clinical_category"].isin(
    C2_CATEGORIES
).all()

assert "EXCLUDED_INJURY" not in set(
    codebook_v1["clinical_category"]
)

# 6. Confirm complete candidate disposition
assert len(reviewed_codes) == 15
assert (~reviewed_codes["c2_include"]).sum() == 3

# Final summary
qa_summary = pd.DataFrame({
    "Check": [
        "Unique ICD code/version pairs",
        "C1 code count",
        "C2 code count",
        "C1 nested within C2",
        "Injury codes excluded",
        "Inventory matching"
    ],
    "Result": [
        "Pass",
        int(codebook_v1["c1_include"].sum()),
        int(codebook_v1["c2_include"].sum()),
        "Pass",
        "Pass",
        "Pass"
    ]
})

display(qa_summary)
print("All technical codebook QA checks passed.")


,Check,Result
0,Unique ICD code/version pairs,Pass
1,C1 code count,5
2,C2 code count,12
3,C1 nested within C2,Pass
4,Injury codes excluded,Pass
5,Inventory matching,Pass


All technical codebook QA checks passed.



### 4.3 Historical ICD & Evidence Verification

**Objective:** Document the historical terminology and evidence basis supporting the selected ICD codes before finalizing the phenotype codebook.

**Method:** Review the twelve selected codes against three types of evidence:

1. **Historical code terminology:** CMS ICD-9-CM and CDC/CMS ICD-10-CM code descriptions applicable to the study period.
2. **Clinical concept alignment:** Consistency between each code's documented meaning and the operational categories defined in Section 3.
3. **Methodological literature:** Published studies motivating transparent, narrow versus expanded administrative case definitions.

**Historical coding context**

MIMIC-IV-ED covers 2011–2019, spanning the U.S. transition to ICD-10-CM on October 1, 2015. Historical code terminology is preferred over current coding definitions.

The ICD-10-CM code `M54.5` was replaced by more specific codes effective October 1, 2021, after the dataset's observation period. These later codes are not included automatically.

**Verification criteria**

Each selected code should have a documented historical reference, a consistent code description, and a clinical rationale supporting its assigned phenotype category.

A successful programmatic match to the MIMIC inventory is not sufficient to establish historical source verification. Verification status must reflect an actual review of the applicable coding source.

This process supports a literature-informed operational phenotype, not independent clinical diagnostic validation.

**Primary coding references**

- CMS. ICD-9-CM Diagnosis and Procedure Codes: Abbreviated and Full Code Titles. https://www.cms.gov/medicare/coding-billing/icd-10-codes/icd-9-cm-diagnosis-procedure-codes-abbreviated-and-full-code-titles
- CDC/NCHS. ICD-10-CM Files (2019 and earlier). https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm


In [7]:

# Historical coding reference registry
ICD_SOURCES = {
    "9": {
        "source": "CMS ICD-9-CM historical code titles",
        "reference_release": "FY2015 (Version 32)",
        "reference_url": (
            "https://www.cms.gov/medicare/coding-billing/"
            "icd-10-codes/icd-9-cm-diagnosis-procedure-codes-"
            "abbreviated-and-full-code-titles"
        )
    },
    "10": {
        "source": "CDC/NCHS ICD-10-CM historical files",
        "reference_release": "FY2016–FY2019",
        "reference_url": (
            "https://archive.cdc.gov/www_cdc_gov/nchs/"
            "icd/comprehensive-listing-icd-10-cm-files.htm"
        )
    }
}

# Create a code-level source verification register
verification_df = codebook_v1[
    [
        "icd_version",
        "icd_code",
        "ed_description",
        "clinical_category",
        "clinical_rationale"
    ]
].copy()

for field in ["source", "reference_release", "reference_url"]:
    verification_df[field] = (
        verification_df["icd_version"]
        .map(lambda v: ICD_SOURCES[v][field])
    )

# An official source has been identified, but each code
# still requires direct historical terminology verification.
verification_df["verification_status"] = "pending_review"

display(verification_df[
    [
        "icd_version",
        "icd_code",
        "ed_description",
        "clinical_category",
        "reference_release",
        "verification_status"
    ]
])

print(f"Codes requiring source review: {len(verification_df)}")


,icd_version,icd_code,ed_description,clinical_category,reference_release,verification_status
0,10,M5430,"Sciatica, unspecified side",SCIATICA_ONLY,FY2016–FY2019,pending_review
1,10,M5431,"Sciatica, right side",SCIATICA_ONLY,FY2016–FY2019,pending_review
2,10,M5432,"Sciatica, left side",SCIATICA_ONLY,FY2016–FY2019,pending_review
3,10,M5440,"Lumbago with sciatica, unspecified side",LBP_WITH_SCIATICA,FY2016–FY2019,pending_review
4,10,M5441,"Lumbago with sciatica, right side",LBP_WITH_SCIATICA,FY2016–FY2019,pending_review
5,10,M5442,"Lumbago with sciatica, left side",LBP_WITH_SCIATICA,FY2016–FY2019,pending_review
6,10,M545,Low back pain,CORE_LBP,FY2016–FY2019,pending_review
7,10,M5489,Other dorsalgia,EXPANDED_BACK_PAIN,FY2016–FY2019,pending_review
8,10,M549,"Dorsalgia, unspecified",EXPANDED_BACK_PAIN,FY2016–FY2019,pending_review
9,9,7242,LUMBAGO,CORE_LBP,FY2015 (Version 32),pending_review


Codes requiring source review: 12



#### Code-Level Verification Decision Record

Each selected ICD code is reviewed against an applicable historical coding reference.

Verification requires:

1. Confirming the code and ICD version in the historical source.
2. Checking whether the official description is consistent with the recorded MIMIC description.
3. Confirming that the code's meaning supports its assigned clinical category.
4. Recording the source release and verification decision.

**Verification status:**

- `verified`: Historical terminology and clinical concept alignment have been manually confirmed.
- `pending_review`: The required source review has not been completed.
- `requires_revision`: A discrepancy or classification concern was identified.

Verification establishes documented source and concept consistency, not independent clinical phenotype validity.


In [8]:

# Create a verification worksheet for manual review
VERIFICATION_PATH = (
    CODEBOOK_DIR / "icd_source_verification.csv"
)

if not VERIFICATION_PATH.exists():
    review_sheet = verification_df[
        [
            "icd_version",
            "icd_code",
            "ed_description",
            "clinical_category",
            "reference_release",
            "reference_url",
            "verification_status"
        ]
    ].copy()

    review_sheet["official_description"] = ""
    review_sheet["review_notes"] = ""

    review_sheet.to_csv(VERIFICATION_PATH, index=False)

    print("Verification worksheet created.")
else:
    print("Existing verification worksheet preserved.")


Existing verification worksheet preserved.


In [9]:

# Load manually reviewed ICD source verification
verified_sources = pd.read_csv(
    CODEBOOK_DIR / "icd_source_verification.csv",
    dtype={"icd_version": "string", "icd_code": "string"}
)

keys = ["icd_version", "icd_code"]

# Confirm complete and unique coverage
assert len(verified_sources) == len(codebook_v1)
assert not verified_sources.duplicated(keys).any()

assert set(map(tuple, verified_sources[keys].to_numpy())) == set(
    map(tuple, codebook_v1[keys].to_numpy())
)

# Check status and evidence fields
assert verified_sources["verification_status"].eq("verified").all()

for col in ["official_description", "reference_release",
            "reference_url", "review_notes"]:
    assert verified_sources[col].notna().all()
    assert verified_sources[col].astype(str).str.strip().ne("").all()

# Verify description agreement, ignoring capitalization
assert (
    verified_sources["ed_description"].str.casefold().str.strip()
    == verified_sources["official_description"].str.casefold().str.strip()
).all()

display(
    verified_sources[
        ["icd_version", "icd_code", "clinical_category",
         "verification_status"]
    ]
)

print("All 12 source verification records are complete.")


,icd_version,icd_code,clinical_category,verification_status
0,10,M5430,SCIATICA_ONLY,verified
1,10,M5431,SCIATICA_ONLY,verified
2,10,M5432,SCIATICA_ONLY,verified
3,10,M5440,LBP_WITH_SCIATICA,verified
4,10,M5441,LBP_WITH_SCIATICA,verified
5,10,M5442,LBP_WITH_SCIATICA,verified
6,10,M545,CORE_LBP,verified
7,10,M5489,EXPANDED_BACK_PAIN,verified
8,10,M549,EXPANDED_BACK_PAIN,verified
9,9,7242,CORE_LBP,verified


All 12 source verification records are complete.



#### Interpretation

All twelve selected ICD codes have documented historical terminology verification records covering ICD-9-CM FY2015 and ICD-10-CM FY2016–FY2020.

The reviewed descriptions are consistent with the corresponding MIMIC-IV-ED inventory descriptions, allowing for capitalization differences. The clinical categories remain aligned with the prespecified operational phenotype rules.

The subsequent FY2022 subdivision of M54.5 was documented as a historical coding change but does not affect the 2011–2019 study population.

These checks support the source traceability of the proposed codebook. They do not constitute independent clinical validation of the LBP phenotypes.



### 4.4 Final Codebook Export

**Objective:** Generate the finalized ICD phenotype codebook by integrating the rule-based clinical classifications with the manually reviewed historical ICD verification records.

**Method:** Merge the technically validated phenotype definitions with the source verification register using the composite key (`icd_version`, `icd_code`). Confirm complete matching, approved verification status, and consistency of recorded clinical categories before export.

The final codebook includes ICD identifiers, clinical descriptions, phenotype categories, C1/C2 membership flags, classification rationale, and evidence verification metadata.

**Output:** `codebooks/icd_lbp_codebook_v1.csv`

This version represents a literature-informed, rule-based operational phenotype with documented historical source verification. It has not undergone independent clinical validation.


In [10]:

# Merge phenotype definitions with manual source verification
keys = ["icd_version", "icd_code"]

verification_fields = [
    *keys,
    "clinical_category",
    "official_description",
    "reference_release",
    "reference_url",
    "verification_status",
    "review_notes"
]

final_codebook = codebook_v1.drop(
    columns=["source_verification_status"]
).merge(
    verified_sources[verification_fields],
    on=keys,
    how="left",
    validate="one_to_one",
    indicator=True,
    suffixes=("", "_review")
)

# Verify complete source matching
assert len(final_codebook) == len(codebook_v1)
assert final_codebook["_merge"].eq("both").all()

# Verify clinical classifications were not changed
assert (
    final_codebook["clinical_category"]
    == final_codebook["clinical_category_review"]
).all()

# Verify manual source approval
assert final_codebook["verification_status"].eq("verified").all()

# Check required evidence fields
for col in [
    "official_description",
    "reference_release",
    "reference_url",
    "review_notes"
]:
    assert final_codebook[col].notna().all()
    assert final_codebook[col].astype(str).str.strip().ne("").all()

# Clean up audit columns
final_codebook = final_codebook.drop(
    columns=["_merge", "clinical_category_review"]
)

final_codebook["codebook_version"] = "v1.0"

# Export
OUTPUT_PATH = CODEBOOK_DIR / "icd_lbp_codebook_v1.csv"

final_codebook.to_csv(OUTPUT_PATH, index=False)

print(f"Final codebook exported: {OUTPUT_PATH.name}")
print(f"Total codes: {len(final_codebook)}")
print(f"C1: {final_codebook['c1_include'].sum()}")
print(f"C2: {final_codebook['c2_include'].sum()}")

display(final_codebook)


Final codebook exported: icd_lbp_codebook_v1.csv
Total codes: 12
C1: 5
C2: 12


,icd_version,icd_code,ed_description,clinical_category,clinical_rationale,c1_include,c2_include,codebook_version,evidence_basis,official_description,reference_release,reference_url,verification_status,review_notes
0,10,M5430,"Sciatica, unspecified side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Sciatica, unspecified side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
1,10,M5431,"Sciatica, right side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Sciatica, right side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
2,10,M5432,"Sciatica, left side",SCIATICA_ONLY,Sciatica without explicit low back pain; expanded phenotype only.,False,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Sciatica, left side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
3,10,M5440,"Lumbago with sciatica, unspecified side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Lumbago with sciatica, unspecified side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
4,10,M5441,"Lumbago with sciatica, right side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Lumbago with sciatica, right side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
5,10,M5442,"Lumbago with sciatica, left side",LBP_WITH_SCIATICA,Explicit lumbago with sciatica.,True,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Lumbago with sciatica, left side",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
6,10,M545,Low back pain,CORE_LBP,Explicit low back pain or lumbago.,True,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,Low back pain,FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,Verified as valid billable code in CDC ICD-10-CM FY2016–FY2020; retired as billable code in FY20...
7,10,M5489,Other dorsalgia,EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,Other dorsalgia,FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and FY2020 releases; exact match."
8,10,M549,"Dorsalgia, unspecified",EXPANDED_BACK_PAIN,Back pain without explicit lumbar localization; expanded phenotype only.,False,True,v1.0,Historical ICD terminology and prespecified clinical concept rules,"Dorsalgia, unspecified",FY2016–FY2020,https://archive.cdc.gov/www_cdc_gov/nchs/icd/comprehensive-listing-icd-10-cm-files.htm,verified,"Verified across CDC ICD-10-CM FY2016, FY2017, FY2018, FY2019, and F


#### Interpretation

The final phenotype codebook was generated by combining the rule-based ICD classification decisions with the manually reviewed historical coding evidence.

All twelve selected codes matched their verification records, and their clinical category assignments remained consistent. The resulting codebook contains five strict LBP codes (C1) and twelve expanded LBP codes (C2), with C1 nested within C2.

The versioned codebook provides a reproducible operational definition for encounter-level cohort construction in P4.

Although historical terminology and classification consistency were reviewed, the phenotype has not undergone independent clinical validation and should not be interpreted as confirmed nonspecific low back pain.


In [11]:
# Confirm exported codebook can be loaded independently
check = pd.read_csv(
    CODEBOOK_DIR / "icd_lbp_codebook_v1.csv",
    dtype={"icd_version": "string", "icd_code": "string"}
)

assert len(check) == 12
assert check["c1_include"].sum() == 5
assert check["c2_include"].sum() == 12

print("Final codebook reload check passed.")

Final codebook reload check passed.



## 5. Final Phenotype Definitions & Phase 4 Handoff

### 5.1 Final Phenotype Definitions

This notebook constructed two literature-informed, rule-based ICD phenotypes for identifying low back pain (LBP)-related emergency department encounters in MIMIC-IV-ED v2.2.

Starting from the database-wide ICD diagnosis inventory, fifteen candidate codes were identified using prespecified symptom-related screening terms. A targeted clinical concept review classified twelve codes as eligible for the expanded LBP phenotype and excluded three injury-specific codes.

The resulting phenotype definitions are:

| Phenotype | Clinical Concept | ICD Codes | Research Role |
|---|---|---|---|
| **C1 — Strict LBP** | Explicit low back pain or lumbago, including LBP with sciatica | 5 | Primary prediction cohort |
| **C2 — Expanded LBP** | C1 plus sciatica-only and broader back pain symptoms | 12 | Cohort sensitivity analysis |
| **C2-only** | Additional codes not included in C1 | 7 | Characterization of the expanded population |

The selected codes include three ICD-9-CM and nine ICD-10-CM codes, reflecting the historical coding systems represented in the dataset.

The codebook passed technical checks for unique ICD identifiers, inventory matching, classification consistency, and C1/C2 nesting. Historical coding terminology was reviewed using applicable ICD-9-CM and ICD-10-CM references, and the review decisions were documented separately.

The finalized operational codebook is stored at:

`codebooks/icd_lbp_codebook_v1.csv`

The corresponding human-reviewed source record is maintained at:

`codebooks/icd_source_verification.csv`

These files document the operational phenotype definitions and their supporting coding evidence.

### 5.2 Methodological Limitations

**ICD-based identification is not clinical confirmation.** The selected codes represent coded symptoms or presentations and cannot establish clinically confirmed nonspecific LBP.

**Candidate screening is not exhaustive.** The initial keyword search was designed to identify a manageable set of relevant symptom codes. Codes describing LBP through other terminology may not have been captured.

**Historical coding differences remain relevant.** ICD-9-CM and ICD-10-CM use different code structures. The analysis preserved the original coding system and documented historical terminology, including the later subdivision of ICD-10-CM `M54.5`.

**Underlying pathology is not fully excluded.** An encounter may contain both an included symptom code and additional diagnoses indicating a specific underlying condition. The codebook does not automatically exclude these encounters.

**Independent clinical validation was not performed.** The study does not estimate sensitivity, specificity, or positive predictive value of the ICD phenotype against chart-based clinical diagnoses.

Accordingly, C1 and C2 should be interpreted as reproducible operational cohort definitions rather than clinically validated diagnostic classifications.

### 5.3 Phase 4 Input Contract — Encounter-Level Cohort Construction

The final ICD codebook will be used in P4 to identify C1 and C2 ED encounters through the MIMIC-IV-ED `diagnosis` table.

**Required codebook fields**

| Field | Purpose |
|---|---|
| `icd_version` | Distinguish ICD-9-CM and ICD-10-CM |
| `icd_code` | Match the recorded diagnosis code |
| `clinical_category` | Describe the assigned clinical concept |
| `c1_include` | Identify strict LBP codes |
| `c2_include` | Identify expanded LBP codes |
| `codebook_version` | Preserve phenotype definition version |

**Cohort construction rules**

1. Match diagnosis records using the composite key (`icd_version`, `icd_code`).
2. Aggregate matching diagnosis records to the ED encounter level (`stay_id`).
3. Assign `c1_flag = 1` if an encounter contains at least one C1 code.
4. Assign `c2_flag = 1` if an encounter contains at least one C2 code.
5. Define `c2_only_flag = 1` when `c2_flag = 1` and `c1_flag = 0`.
6. Preserve exactly one analytical record per `stay_id`.
7. Determine adult eligibility and finalize unique encounter and patient counts in P4.

An encounter with multiple qualifying diagnosis codes must be counted only once within each cohort.

C1 is nested within C2, and the same cohort eligibility principles will be applied when comparing downstream model performance.

Final ICD codes are used for retrospective cohort identification and must not be included as triage-time predictive features.

**P4 deliverables**

- Encounter-level C1 and C2 membership flags.
- Adult-eligible cohort counts and patient counts.
- Characterization of C2-only encounters.
- A documented, reproducible cohort integration process.

These deliverables will support subsequent opioid dispensing outcome construction and predictive modeling.

### Conclusion

The ICD phenotyping workflow produced two transparent, reproducible operational definitions of LBP-related ED encounters, supported by documented clinical concept rules, historical ICD source review, and technical quality assurance.

The resulting codebook provides a consistent foundation for evaluating how phenotype definition influences cohort composition, documented opioid dispensing prevalence, and predictive model performance.

The phenotypes remain ICD-based research proxies and are not intended to establish clinically confirmed diagnoses.
